## Import

In [1]:
import os
for _v in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ[_v] = "1"   # single-threaded so parallel jobs do not fight over cores

import json, os, time, pickle, warnings
import numpy as np
import pandas as pd
import optuna
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, log_loss, average_precision_score)
warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

/Users/waroonragwongsiri/Playground/IntroMLHomework/Data/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Setup

In [2]:
SEED = 42
N_TRIALS = 30
INNER_K = 3
np.random.seed(SEED)
os.makedirs("weights_V5_0", exist_ok=True)

In [3]:
df = pd.read_csv("marketing_campaign.csv", sep="\t")
df = df.drop(columns=["ID", "Z_CostContact", "Z_Revenue"])
df["Dt_Customer"] = pd.to_datetime(df["Dt_Customer"], format="%d-%m-%Y")
ref_date = df["Dt_Customer"].max()
df["Customer_Tenure_Days"] = (ref_date - df["Dt_Customer"]).dt.days
df["Age"] = 2014 - df["Year_Birth"]
df = df[~df["Year_Birth"].isin([1893, 1899, 1900])].copy()
df = df[df["Income"] != 666666].copy()  # drop $666,666 income outlier (same treatment as Year_Birth sentinels)
df["Marital_Status"] = df["Marital_Status"].replace({"Absurd": "Single", "YOLO": "Single", "Alone": "Single"})
print("cleaned:", df.shape, "| Response rate:", round(df["Response"].mean(), 4))

CLF_NUMERIC = ["AcceptedCmp5","AcceptedCmp3","MntWines","MntMeatProducts","NumCatalogPurchases",
    "Recency","Customer_Tenure_Days","MntGoldProds","Income","MntFruits","AcceptedCmp1",
    "AcceptedCmp2","NumWebPurchases","MntSweetProducts","MntFishProducts","NumStorePurchases","Age"]
X_num = df[CLF_NUMERIC].copy()
X_cat = pd.get_dummies(df[["Marital_Status"]], dtype=float)
X_full = pd.concat([X_num, X_cat], axis=1)
FEATURE_COLS = list(X_full.columns)
y_full = df["Response"].astype(float).values
print(f"features: {len(CLF_NUMERIC)} numeric + {X_cat.shape[1]} dummies = {len(FEATURE_COLS)} dims")

cleaned: (2236, 28) | Response rate: 0.1494
features: 17 numeric + 5 dummies = 22 dims


## Feature Interaction

In [4]:
# Feature interaction hand-off (Feature Interaction section of classify.ipynb):
# the strongest interacting features (Teenhome, NumDealsPurchases, NumWebVisitsMonth) are not in the 22-dim model set.
EXTRA = ["Teenhome", "NumDealsPurchases", "NumWebVisitsMonth"]
X_full = pd.concat([X_full, df[EXTRA].astype(float)], axis=1)
ALL_COLS = list(X_full.columns)
BASE_COLS = ALL_COLS[:len(FEATURE_COLS)]
X_all = X_full.values
y_all = y_full.astype(int)

def top_pairs(csv_path, cols, k):
    """Top-k significant (q < 0.05) pairs by effect size where both features are in `cols`."""
    t = pd.read_csv(csv_path)
    t = t[(t.q < 0.05) & t.A.isin(cols) & t.B.isin(cols)].sort_values("effect", ascending=False)
    return list(zip(t.A.head(k), t.B.head(k)))

PAIRS = top_pairs("results/interaction_clf.csv", BASE_COLS + EXTRA, 5)
print("pair features:", PAIRS)
SETS = {
    "base (22)":                  dict(cols=BASE_COLS, pairs=[]),
    "+ 3 modifiers (25)":         dict(cols=BASE_COLS + EXTRA, pairs=[]),
    "+ 3 modifiers + 5 pairs":    dict(cols=BASE_COLS + EXTRA, pairs=PAIRS),
}
inc_name = "Income"

def prep_split(tr_idx, te_idx, set_name):
    """Median-fill Income, standardise, then append standardised products for the pair features (training rows only)."""
    cols, pairs = SETS[set_name]["cols"], SETS[set_name]["pairs"]
    ci = [ALL_COLS.index(c) for c in cols]
    A, B = X_all[tr_idx][:, ci].copy(), X_all[te_idx][:, ci].copy()
    j = cols.index(inc_name); med = np.nanmedian(A[:, j])
    for a in (A, B):
        a[:, j] = np.where(np.isnan(a[:, j]), med, a[:, j])
    sc = StandardScaler().fit(A); A, B = sc.transform(A), sc.transform(B)
    if pairs:
        ix = {c: i for i, c in enumerate(cols)}
        PA = np.column_stack([A[:, ix[a]] * A[:, ix[b]] for a, b in pairs])
        PB = np.column_stack([B[:, ix[a]] * B[:, ix[b]] for a, b in pairs])
        sp = StandardScaler().fit(PA); A, B = np.hstack([A, sp.transform(PA)]), np.hstack([B, sp.transform(PB)])
    return A, B, y_all[tr_idx], y_all[te_idx]

idx_all = np.arange(len(y_all))
tr_idx, te_idx = train_test_split(idx_all, test_size=0.20, random_state=SEED, stratify=y_all)
for s_ in SETS:
    a_, b_, _, _ = prep_split(tr_idx, te_idx, s_)
    print(f"{s_:26s} train {a_.shape} test {b_.shape}")

pair features: [('Teenhome', 'NumWebVisitsMonth'), ('NumDealsPurchases', 'NumStorePurchases'), ('Income', 'NumDealsPurchases'), ('NumDealsPurchases', 'NumWebVisitsMonth'), ('MntSweetProducts', 'NumStorePurchases')]
base (22)                  train (1788, 22) test (448, 22)
+ 3 modifiers (25)         train (1788, 25) test (448, 25)
+ 3 modifiers + 5 pairs    train (1788, 30) test (448, 30)


## Model

In [5]:
import lightgbm as lgb
POS_W = float((y_all == 0).sum() / (y_all == 1).sum())  # class ratio, one scalar
SPECS = {
 "LightGBM": dict(tier="curriculum", kind="proba",
    space=lambda t: dict(n=t.suggest_int("n", 100, 600, step=100), lr=t.suggest_float("lr", 0.01, 0.3, log=True),
                         nl=t.suggest_int("nl", 4, 64), mcs=t.suggest_int("mcs", 5, 50),
                         sub=t.suggest_float("sub", 0.6, 1.0), col=t.suggest_float("col", 0.5, 1.0),
                         lam=t.suggest_float("lam", 1e-2, 10, log=True), spw=t.suggest_categorical("spw", [1.0, POS_W])),
    make=lambda p, s: lgb.LGBMClassifier(n_estimators=p["n"], learning_rate=p["lr"], num_leaves=p["nl"],
                         min_child_samples=p["mcs"], subsample=p["sub"], subsample_freq=1, colsample_bytree=p["col"],
                         reg_lambda=p["lam"], scale_pos_weight=p["spw"], n_jobs=1, random_state=s, verbosity=-1)),
}

In [6]:
def score_of(est, X, kind):
    return est.predict_proba(X)[:, 1] if kind == "proba" else est.decision_function(X)

def inner_oof(spec, params, X, y, seed):
    """Out-of-fold scores on the training rows (inner K-fold)."""
    oof = np.zeros(len(y))
    for a, b in StratifiedKFold(INNER_K, shuffle=True, random_state=seed).split(X, y):
        est = spec["make"](params, seed)
        est.fit(X[a], y[a])
        oof[b] = score_of(est, X[b], spec["kind"])
    return oof

def tune(spec, X, y, seed):
    """Optuna (TPE) maximising average precision of the inner out-of-fold scores."""
    def objective(trial):
        return average_precision_score(y, inner_oof(spec, spec["space"](trial), X, y, seed))
    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=seed))
    study.optimize(objective, n_trials=spec.get("trials", N_TRIALS))
    return spec["space"](optuna.trial.FixedTrial(study.best_params))

def best_threshold(y, s):
    """F1-maximising threshold, searched on training-side out-of-fold scores only."""
    cand = np.unique(np.quantile(s, np.linspace(0.40, 0.99, 120)))
    f = [f1_score(y, (s >= t).astype(int), zero_division=0) for t in cand]
    return float(cand[int(np.argmax(f))])

def fit_eval(spec, params, Xa, ya, Xb, yb, seed):
    """Threshold from inner OOF on (Xa, ya); refit on all of (Xa, ya); score (Xb, yb)."""
    thr = best_threshold(ya, inner_oof(spec, params, Xa, ya, seed))
    est = spec["make"](params, seed)
    est.fit(Xa, ya)
    s = score_of(est, Xb, spec["kind"])
    pred = (s >= thr).astype(int)
    row = {"f1": f1_score(yb, pred, zero_division=0), "accuracy": accuracy_score(yb, pred),
           "precision": precision_score(yb, pred, zero_division=0),
           "recall": recall_score(yb, pred, zero_division=0),
           "roc_auc": roc_auc_score(yb, s), "threshold": thr}
    if spec["kind"] == "proba":
        row["bce"] = log_loss(yb, np.clip(s, 1e-12, 1 - 1e-12))
    return row, s

## Sanity Check

In [7]:
a, b = next(StratifiedKFold(5, shuffle=True, random_state=SEED).split(X_all, y_all))
assert len(set(a) & set(b)) == 0, "train / eval rows overlap"
for sname, spec in SETS.items():
    Xa, Xb, ya, yb = prep_split(a, b, sname)
    want = len(spec["cols"]) + len(spec["pairs"])
    assert Xa.shape[1] == Xb.shape[1] == want, (sname, Xa.shape, want)
    assert np.allclose(Xa.mean(0), 0, atol=1e-6) and np.allclose(Xa.std(0), 1, atol=1e-6), "train columns must be standardised"
    assert not np.allclose(Xb.mean(0), 0, atol=1e-6), "eval rows must not be re-centred (scaler is fitted on train only)"
    assert np.isfinite(Xa).all() and np.isfinite(Xb).all()
    print(f"{sname:26s} dims {Xa.shape[1]:3d} | train mean 0, std 1 | eval mean not 0 -> OK")

base (22)                  dims  22 | train mean 0, std 1 | eval mean not 0 -> OK
+ 3 modifiers (25)         dims  25 | train mean 0, std 1 | eval mean not 0 -> OK


+ 3 modifiers + 5 pairs    dims  30 | train mean 0, std 1 | eval mean not 0 -> OK


## Training

In [8]:
results, scores, best_params = [], {}, {}
spec = SPECS["LightGBM"]
for sname in SETS:
    Xa, Xb, ya, yb = prep_split(tr_idx, te_idx, sname)
    t0 = time.time()
    params = tune(spec, Xa, ya, SEED)
    row, s = fit_eval(spec, params, Xa, ya, Xb, yb, SEED)
    best_params[sname] = params; scores[sname] = s
    results.append(dict(model=f"LightGBM | {sname}", tier="curriculum", selection="inner validation",
                        **{k: round(float(v), 4) for k, v in row.items()}, params=params))
    print(f"{sname:32s} F1 = {row['f1']:.4f} | {time.time() - t0:.0f}s")
json.dump(best_params, open("weights_V5_0/classification_lightgbm_best_params.json", "w"), indent=2, default=str)

base (22)                        F1 = 0.5344 | 36s


+ 3 modifiers (25)               F1 = 0.5799 | 31s


+ 3 modifiers + 5 pairs          F1 = 0.6061 | 27s


## Evaluation

In [9]:
table = pd.DataFrame(results)
print(table.drop(columns=[c for c in ["tier", "selection", "params"] if c in table]).to_string(index=False))
json.dump({"n_test": int(len(yte_ := prep_split(tr_idx, te_idx, next(iter(SETS)))[3])), "seed": SEED, "models": results},
          open("weights_V5_0/classification_lightgbm_test_metrics.json", "w"), indent=2, default=str)
np.savez("weights_V5_0/classification_lightgbm_test_scores.npz", y_true=yte_, **scores)

                             model     f1  accuracy  precision  recall  roc_auc  threshold    bce
              LightGBM | base (22) 0.5344    0.8638     0.5469  0.5224   0.8859     0.6041 0.3154
     LightGBM | + 3 modifiers (25) 0.5799    0.8415     0.4804  0.7313   0.9030     0.4126 0.2964
LightGBM | + 3 modifiers + 5 pairs 0.6061    0.8839     0.6154  0.5970   0.8954     0.6235 0.3193
